# AI-Powered Air Quality Analytics & Pollution Prediction System
**BharatCares–AICTE Internship in Collaboration with IBM**  
**Domain**: Data Analytics, Environmental Informatics & Machine Learning  
**Target Output**: 24-Hour Ahead National Air Quality Index (AQI $t+24$) Prediction

---

## 1. Project Introduction
Urban air pollution is one of the most critical public health and environmental challenges confronting modern civilization, particularly across South Asia. Rapid industrialization, high vehicular density, seasonal agricultural residue burning, and geographical topographies have led to hazardous concentrations of respirable particulate matter ($PM_{2.5}$, $PM_{10}$) and toxic gaseous pollutants ($NO_2$, $SO_2$, $CO$, $O_3$).

This project delivers an end-to-end analytical and artificial intelligence framework designed to ingest, clean, explore, model, and forecast air quality dynamics across major metropolitan centers in India using official historical records published by the **Central Pollution Control Board (CPCB)**.


## 2. Problem Statement
Air Quality Index (AQI) values display extreme non-linear volatility driven by complex meteorological forcing, diurnal anthropogenic emission cycles, and seasonal atmospheric inversions. Traditional static threshold monitoring systems fail to provide proactive warnings before hazardous pollution episodes unfold.

### Core Research Questions:
1. Which Indian cities endure the highest historical average AQI and chronic exposure?
2. Which months and calendar windows exhibit recurring peak pollution episodes?
3. How do atmospheric seasons (Winter, Summer, Monsoon, Post-Monsoon) modulate air quality?
4. Which individual chemical pollutants exhibit the strongest empirical correlation with composite AQI?
5. How has national air quality evolved over multi-year horizons (2015–2020)?
6. Are recurring pollution spikes episodic or structurally continuous?
7. Which cities suffer from the greatest atmospheric volatility and dispersion instability?


## 3. Project Objectives & System Architecture
The primary technical and analytical objectives of this project are:
* **Data Pipeline Engineering**: Construct a robust ingestion and preprocessing pipeline capable of handling missing sensor values and physical bound anomalies without deleting genuine extreme episodic hazards.
* **Exploratory Environmental Analytics**: Answer the 7 core environmental research questions using statistical hypothesis testing and rich data visualizations.
* **Leak-Free Feature Engineering**: Engineer historical lags, rolling statistics, calendar indicators, and pollutant ratios strictly observing temporal precedence ($t \le 	ext{observation time}$).
* **Predictive Modeling**: Formulate and benchmark candidate models (Persistence Baseline, Linear Regression, Random Forest Regressor, XGBoost) to forecast AQI 24 hours in advance ($t + 24\text{ hours}$).
* **Strict Chronological Evaluation**: Partition data into Train (70%), Validation (15%), and Test (15%) chronological splits to guarantee that no future data leaks into model training.
* **Data-Driven Automated Insights**: Generate mathematically grounded observations explaining seasonal increases, geographic disparities, and primary pollutant drivers.


## 4. Dataset Description & Provenance
The empirical data used in this study originates from the **Central Pollution Control Board (CPCB)** ambient air quality monitoring network, consolidated into daily urban records across 26 major Indian cities.

### Feature Inventory:
* `City`: Metropolitan jurisdiction (e.g., Delhi, Bengaluru, Mumbai, Chennai, Kolkata)
* `Date`: Observation timestamp (daily granularity, 2015 to 2020)
* `PM2.5`, `PM10`: Particulate matter concentration ($\mu g/m^3$)
* `NO`, `NO2`, `NOx`, `NH3`: Nitrogen compounds and ammonia ($\mu g/m^3$ / ppb)
* `CO`: Carbon monoxide ($mg/m^3$)
* `SO2`: Sulfur dioxide ($\mu g/m^3$)
* `O3`: Ground-level ozone ($\mu g/m^3$)
* `Benzene`, `Toluene`, `Xylene`: Volatile organic compounds (VOCs)
* `AQI`: Official National Air Quality Index score (0 to 500+)
* `AQI_Bucket`: Qualitative health category (Good, Satisfactory, Moderate, Poor, Very Poor, Severe)


## 5. Library Installation and Environment Setup
We install and import standard data science and machine learning libraries.

In [ ]:
# Install required dependencies for Google Colab environment
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly xgboost scipy

import os
import sys
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Optional XGBoost import
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# Visualization styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print("Libraries successfully initialized!")


## 6. Data Loading
We load the official CPCB historical dataset (`city_day.csv`). If running in Google Colab or an environment without the local file, we automatically download it from our verified mirror.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/adityarc19/aqi-india/main/city_day.csv"
LOCAL_FILE = "city_day.csv"

if not os.path.exists(LOCAL_FILE):
    if os.path.exists("../data/city_day.csv"):
        LOCAL_FILE = "../data/city_day.csv"
    elif os.path.exists("data/city_day.csv"):
        LOCAL_FILE = "data/city_day.csv"
    else:
        print(f"Downloading dataset from verified mirror: {DATA_URL}")
        req = urllib.request.Request(DATA_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open("city_day.csv", "wb") as f:
            f.write(resp.read())
        LOCAL_FILE = "city_day.csv"

raw_df = pd.read_csv(LOCAL_FILE)
print(f"Dataset Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print(f"Unique Cities ({raw_df['City'].nunique()}): {list(raw_df['City'].unique()[:8])}...")
raw_df.head(5)


## 7. Data Cleaning & Preprocessing Decisions
We implement systematic preprocessing:
1. **Date Standardisation**: Parse observation date to `datetime64[ns]` and sort chronologically by `(City, Date)`.
2. **Deduplication**: Remove duplicate records sharing identical City and Date timestamps.
3. **Physical Constraints**: Replace negative sensor recordings with `NaN`.
4. **Outlier Policy**: Compute IQR statistics. Authentic episodic surges (e.g. crop burning, winter inversion) are preserved rather than trimmed, as they represent genuine environmental hazards that the model must learn.
5. **Spatial-Temporal Imputation**: Fill short sensor dropouts using city-level forward fill (limit=3 days) followed by city-month median imputation.
6. **Target Verification**: Retain records containing valid AQI labels for supervised regression.


In [ ]:
df_cleaned = raw_df.copy()

# 1. Date conversion & chronological sorting
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'], errors='coerce')
df_cleaned = df_cleaned.dropna(subset=['Date']).sort_values(by=['City', 'Date']).reset_index(drop=True)

# 2. Remove duplicates
initial_len = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates(subset=['City', 'Date'], keep='first').reset_index(drop=True)
print(f"Dropped {initial_len - len(df_cleaned)} duplicate rows.")

# 3. Physical bounds validation (negatives -> NaN)
numeric_cols = [c for c in df_cleaned.columns if c not in ['City', 'Date', 'AQI_Bucket']]
for col in numeric_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    neg_count = (df_cleaned[col] < 0).sum()
    if neg_count > 0:
        df_cleaned.loc[df_cleaned[col] < 0, col] = np.nan

# 4. Temporal & City-Aware Imputation
# Forward fill up to 3 days per city
df_cleaned[numeric_cols] = df_cleaned.groupby('City')[numeric_cols].transform(
    lambda s: s.ffill(limit=3).bfill(limit=1)
)

# Month-City median imputation for remaining missing pollutants
df_cleaned['_Month'] = df_cleaned['Date'].dt.month
for col in numeric_cols:
    if df_cleaned[col].isnull().any():
        city_month_med = df_cleaned.groupby(['City', '_Month'])[col].transform('median')
        df_cleaned[col] = df_cleaned[col].fillna(city_month_med)
        if df_cleaned[col].isnull().any():
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
df_cleaned = df_cleaned.drop(columns=['_Month'])

# Drop records missing AQI target
df_cleaned = df_cleaned.dropna(subset=['AQI']).reset_index(drop=True)

# CPCB AQI Bucket mapping
bins = [-np.inf, 50, 100, 200, 300, 400, np.inf]
labels = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
df_cleaned['AQI_Bucket'] = pd.cut(df_cleaned['AQI'], bins=bins, labels=labels)

print(f"Preprocessed Dataset: {len(df_cleaned):,} rows across {df_cleaned['City'].nunique()} cities.")
print("Missing values per column after cleaning:")
print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])


## 8. Feature Engineering (Strictly Anti-Leakage)
To predict AQI 24 hours into the future, we define the target:
$$\text{Target\_AQI\_t24} = \text{AQI}_{t+1\text{ day}} = \text{shift}(-1)$$

All input features must strictly use information observed at time $t$ or earlier:
* **Calendar Features**: `Year`, `Month`, `Day`, `DayOfWeek`, `Is_Weekend`, and cyclical $\sin / \cos$ transformations of month and day of week.
* **Meteorological Season**: Categorized according to Indian meteorological standards (Winter, Summer, Monsoon, Post-Monsoon).
* **Historical Lags**: Immediate observed $\text{AQI}_t$, $\text{AQI}_{t-1}$, $\text{AQI}_{t-2}$, $\text{AQI}_{t-3}$, and weekly lag $\text{AQI}_{t-7}$. Lags of primary pollutants ($PM_{2.5}, PM_{10}, NO_2, SO_2, CO, O_3$).
* **Past Rolling Statistics**: 7-day and 14-day rolling mean, standard deviation (volatility), minimum, and maximum of AQI, strictly computed on past windows.
* **Domain Ratios**: $PM_{2.5} / PM_{10}$ ratio (combustion vs crustal dust marker) and $NO_2 / SO_2$ ratio (vehicular vs industrial emission marker).


In [ ]:
def engineer_features(df):
    data = df.copy().sort_values(by=['City', 'Date']).reset_index(drop=True)
    
    # Calendar & Cyclical features
    data['Year'] = data['Date'].dt.year
    data['Month'] = data['Date'].dt.month
    data['Day'] = data['Date'].dt.day
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['Is_Weekend'] = (data['DayOfWeek'] >= 5).astype(int)
    
    def get_season(m):
        if m in [12, 1, 2]: return 'Winter'
        elif m in [3, 4, 5]: return 'Summer'
        elif m in [6, 7, 8, 9]: return 'Monsoon'
        else: return 'Post-Monsoon'
        
    data['Season'] = data['Month'].apply(get_season)
    data['Month_Sin'] = np.sin(2 * np.pi * data['Month'] / 12.0)
    data['Month_Cos'] = np.cos(2 * np.pi * data['Month'] / 12.0)
    data['DayOfWeek_Sin'] = np.sin(2 * np.pi * data['DayOfWeek'] / 7.0)
    data['DayOfWeek_Cos'] = np.cos(2 * np.pi * data['DayOfWeek'] / 7.0)
    
    # Lagged features (historical up to time t)
    data['AQI_t'] = data['AQI']
    data['AQI_lag1'] = data.groupby('City')['AQI'].shift(1)
    data['AQI_lag2'] = data.groupby('City')['AQI'].shift(2)
    data['AQI_lag3'] = data.groupby('City')['AQI'].shift(3)
    data['AQI_lag7'] = data.groupby('City')['AQI'].shift(7)
    
    core_pols = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
    for pol in core_pols:
        if pol in data.columns:
            data[f'{pol}_t'] = data[pol]
            data[f'{pol}_lag1'] = data.groupby('City')[pol].shift(1)
            
    # Rolling statistics strictly on historical observations
    data['AQI_roll_mean_7'] = data.groupby('City')['AQI'].transform(lambda s: s.rolling(7, min_periods=2).mean())
    data['AQI_roll_std_7'] = data.groupby('City')['AQI'].transform(lambda s: s.rolling(7, min_periods=2).std().fillna(0.0))
    data['AQI_roll_mean_14'] = data.groupby('City')['AQI'].transform(lambda s: s.rolling(14, min_periods=3).mean())
    data['AQI_roll_max_7'] = data.groupby('City')['AQI'].transform(lambda s: s.rolling(7, min_periods=2).max())
    data['AQI_roll_min_7'] = data.groupby('City')['AQI'].transform(lambda s: s.rolling(7, min_periods=2).min())
    
    if 'PM2.5' in data.columns:
        data['PM2.5_roll_mean_7'] = data.groupby('City')['PM2.5'].transform(lambda s: s.rolling(7, min_periods=2).mean())
    if 'PM2.5' in data.columns and 'PM10' in data.columns:
        data['PM_Ratio_t'] = (data['PM2.5'] / (data['PM10'] + 1e-5)).clip(0.0, 1.0)
    if 'NO2' in data.columns and 'SO2' in data.columns:
        data['NO2_SO2_Ratio_t'] = (data['NO2'] / (data['SO2'] + 1e-5)).clip(0.0, 50.0)
        
    # Define Target: Next Day AQI (t + 24 hours)
    data['Target_AQI_t24'] = data.groupby('City')['AQI'].shift(-1)
    return data

feat_df = engineer_features(df_cleaned)

# Identify modeling features
excluded = ['Date', 'City', 'AQI', 'Target_AQI_t24', 'AQI_Bucket', 'Season']
feature_cols = [c for c in feat_df.columns if c not in excluded]

# Fill warmup NaNs within city
for c in feature_cols:
    feat_df[c] = feat_df.groupby('City')[c].transform(lambda s: s.bfill().ffill())
    feat_df[c] = feat_df[c].fillna(feat_df[c].median())

# Drop rows where future target is missing (e.g. final observation per city)
ml_data = feat_df.dropna(subset=['Target_AQI_t24']).reset_index(drop=True)
print(f"Modeling Matrix: {ml_data.shape[0]:,} rows x {len(feature_cols)} features")
print(f"Features ({len(feature_cols)}): {feature_cols}")


## 9. Exploratory Data Analysis (EDA)
We systematically investigate the 7 core environmental research questions using high-resolution statistical visualizations.

In [ ]:
# Setup figure canvas for comprehensive EDA
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# 1. City-Wise Mean AQI
top_cities = df_cleaned.groupby('City')['AQI'].agg(['mean', 'std']).sort_values(by='mean', ascending=False).head(10)
axes[0, 0].barh(top_cities.index[::-1], top_cities['mean'][::-1], xerr=top_cities['std'][::-1], color='#e63946', alpha=0.85, capsize=4)
axes[0, 0].set_title("1. Top 10 Polluted Cities by Mean AQI (±1 Std Dev)")
axes[0, 0].set_xlabel("Mean AQI Score")

# 2. Monthly Pollution Profile
df_cleaned['Month'] = df_cleaned['Date'].dt.month
monthly_aqi = df_cleaned.groupby('Month')['AQI'].mean()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[0, 1].plot(month_labels, monthly_aqi, marker='o', color='#457b9d', linewidth=2.5, markersize=7)
axes[0, 1].fill_between(range(12), monthly_aqi, color='#a8dadc', alpha=0.4)
axes[0, 1].set_title("2. Monthly Mean AQI Cycle (Winter Peaks vs Monsoon Dip)")
axes[0, 1].set_ylabel("Mean AQI")

# 3. Seasonal AQI Boxplots
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
sns.boxplot(data=feat_df, x='Season', y='AQI', order=season_order, palette='Set2', ax=axes[1, 0], showfliers=False)
axes[1, 0].set_title("3. Seasonal AQI Distributions")
axes[1, 0].set_ylabel("AQI Score")

# 4. Pollutant Correlation with AQI
pol_list = [c for c in ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3'] if c in df_cleaned.columns]
corrs = df_cleaned[['AQI'] + pol_list].corr()['AQI'].drop('AQI').sort_values(ascending=True)
axes[1, 1].barh(corrs.index, corrs.values, color='#2a9d8f')
axes[1, 1].set_title("4. Correlation of Pollutants with Overall AQI (Pearson r)")
axes[1, 1].set_xlabel("Pearson Correlation Coefficient")

# 5. Multi-Year AQI Trajectory
yearly_aqi = df_cleaned.groupby(df_cleaned['Date'].dt.year)['AQI'].mean()
axes[2, 0].plot(yearly_aqi.index, yearly_aqi.values, marker='s', color='#f4a261', linewidth=2.5, markersize=8)
axes[2, 0].set_title("5. Multi-Year National AQI Progression (2015-2020)")
axes[2, 0].set_xlabel("Year")
axes[2, 0].set_ylabel("Average AQI")

# 6. CPCB Category Frequency Breakdown
cat_counts = df_cleaned['AQI_Bucket'].value_counts()
palette = ['#009966', '#80c000', '#ff9933', '#cc0033', '#660099', '#7e0023']
axes[2, 1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%', startangle=140, colors=palette[:len(cat_counts)])
axes[2, 1].set_title("6. Overall CPCB Health Category Share")

plt.tight_layout()
plt.show()


## 10. Statistical Hypothesis Testing
To verify whether seasonal pollution variations are statistically significant rather than random fluctuations, we formulate a two-sample hypothesis test:
* **Null Hypothesis ($H_0$)**: There is no significant difference between Winter AQI and Monsoon AQI.
* **Alternative Hypothesis ($H_1$)**: Winter AQI is significantly higher than Monsoon AQI due to atmospheric trapping.

In [ ]:
winter_aqi = feat_df[feat_df['Season'] == 'Winter']['AQI'].dropna()
monsoon_aqi = feat_df[feat_df['Season'] == 'Monsoon']['AQI'].dropna()

# Mann-Whitney U test (non-parametric, robust to non-normality)
u_stat, p_val = stats.mannwhitneyu(winter_aqi, monsoon_aqi, alternative='greater')

print("=== STATISTICAL HYPOTHESIS TEST RESULTS ===")
print(f"Winter Mean AQI: {winter_aqi.mean():.2f} (n={len(winter_aqi)})")
print(f"Monsoon Mean AQI: {monsoon_aqi.mean():.2f} (n={len(monsoon_aqi)})")
print(f"Mann-Whitney U Statistic: {u_stat:,.1f}")
print(f"p-value: {p_val:.2e}")

if p_val < 0.001:
    print("CONCLUSION: Reject H0 with extreme statistical confidence (p < 0.001). Winter pollution is significantly elevated compared to Monsoon.")
else:
    print("CONCLUSION: Fail to reject H0.")


## 11. Machine Learning Formulation & Chronological Splitting
Time-series prediction requires a strict chronological train-validation-test split to ensure realistic evaluation without future lookahead leakage.
* **Train (70%)**: Earliest observation dates (used exclusively to fit preprocessing scalers and learn model weights).
* **Validation (15%)**: Intermediate chronological dates (used to tune hyperparameters and benchmark candidate architectures).
* **Test (15%)**: Final holdout period (untouched until final champion model reporting).


In [ ]:
# Chronological splitting
sorted_dates = np.sort(ml_data['Date'].unique())
n_dates = len(sorted_dates)
train_cutoff = sorted_dates[int(n_dates * 0.70)]
val_cutoff = sorted_dates[int(n_dates * 0.85)]

train_set = ml_data[ml_data['Date'] < train_cutoff].reset_index(drop=True)
val_set = ml_data[(ml_data['Date'] >= train_cutoff) & (ml_data['Date'] < val_cutoff)].reset_index(drop=True)
test_set = ml_data[ml_data['Date'] >= val_cutoff].reset_index(drop=True)

print(f"Train Partition: {train_set['Date'].min().date()} to {train_set['Date'].max().date()} ({len(train_set):,} samples)")
print(f"Validation Partition: {val_set['Date'].min().date()} to {val_set['Date'].max().date()} ({len(val_set):,} samples)")
print(f"Test Partition: {test_set['Date'].min().date()} to {test_set['Date'].max().date()} ({len(test_set):,} samples)")

X_train, y_train = train_set[feature_cols], train_set['Target_AQI_t24'].values
X_val, y_val = val_set[feature_cols], val_set['Target_AQI_t24'].values
X_test, y_test = test_set[feature_cols], test_set['Target_AQI_t24'].values

# Feature Scaling fit solely on Training partition
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


## 12. Model Training
We train 4 candidate predictive models:
1. **Persistence Baseline**: $\hat{y}_{t+24} = \text{AQI}_t$ (the last known AQI).
2. **Linear Regression**: Standardized parametric model.
3. **Random Forest Regressor**: Non-linear ensemble model capturing complex non-linear meteorological dynamics.
4. **XGBoost Regressor**: Gradient-boosted decision trees.


In [ ]:
# 1. Persistence Baseline
val_preds = {'Persistence Baseline': X_val['AQI_t'].values}
test_preds = {'Persistence Baseline': X_test['AQI_t'].values}

# 2. Linear Regression
print("Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
val_preds['Linear Regression'] = lr.predict(X_val_scaled)
test_preds['Linear Regression'] = lr.predict(X_test_scaled)

# 3. Random Forest Regressor
print("Training Random Forest Regressor...")
rf = RandomForestRegressor(n_estimators=100, max_depth=14, min_samples_split=4, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
val_preds['Random Forest'] = rf.predict(X_val)
test_preds['Random Forest'] = rf.predict(X_test)

# 4. XGBoost Regressor
if HAS_XGB:
    print("Training XGBoost Regressor...")
    xgb_model = xgb.XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    val_preds['XGBoost'] = xgb_model.predict(X_val)
    test_preds['XGBoost'] = xgb_model.predict(X_test)

print("All candidate models successfully trained!")


## 13. Model Evaluation & Comparison
We evaluate all models on the Validation partition using:
* **MAE (Mean Absolute Error)**: Average magnitude of prediction errors in AQI units.
* **RMSE (Root Mean Squared Error)**: Penalizes large error outliers.
* **$R^2$ Score**: Proportion of variance explained by model.


In [ ]:
def eval_metrics(y_true, y_pred):
    return {
        'MAE': round(float(mean_absolute_error(y_true, y_pred)), 2),
        'RMSE': round(float(root_mean_squared_error(y_true, y_pred)), 2),
        'R2': round(float(r2_score(y_true, y_pred)), 4)
    }

val_results = []
for model_name, preds in val_preds.items():
    m = eval_metrics(y_val, preds)
    m['Model'] = model_name
    val_results.append(m)

val_comparison_df = pd.DataFrame(val_results).sort_values(by='RMSE').reset_index(drop=True)
print("=== VALIDATION SET MODEL COMPARISON ===")
display(val_comparison_df)

# Select champion model
best_name = val_comparison_df.iloc[0]['Model']
if best_name == 'Persistence Baseline':
    best_name = val_comparison_df[val_comparison_df['Model'] != 'Persistence Baseline'].iloc[0]['Model']

print(f"\nChampion Selected: {best_name}")

# Final Untouched Test Evaluation
test_m = eval_metrics(y_test, test_preds[best_name])
print(f"=== UNTOUCHED TEST SET PERFORMANCE ({best_name}) ===")
print(f"MAE:  {test_m['MAE']} AQI points")
print(f"RMSE: {test_m['RMSE']} AQI points")
print(f"R2:   {test_m['R2']}")


## 14. Actual vs Predicted Visualization & Residual Diagnostics
We inspect the calibration of the champion model on the holdout test set.

In [ ]:
champ_preds = test_preds[best_name]
residuals = y_test - champ_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter Plot: Actual vs Predicted
axes[0].scatter(y_test, champ_preds, alpha=0.3, color='#1d3557', s=18)
min_val = min(y_test.min(), champ_preds.min())
max_val = max(y_test.max(), champ_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction (y=x)')
axes[0].set_title(f'Actual vs Predicted AQI (t+24h) — {best_name}')
axes[0].set_xlabel('Actual Future AQI')
axes[0].set_ylabel('Predicted AQI')
axes[0].legend()

# Residual Distribution
sns.histplot(residuals, bins=50, kde=True, color='#e76f51', ax=axes[1])
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title(f'Residual Error Distribution (Mean={residuals.mean():.2f})')
axes[1].set_xlabel('Error (Actual - Predicted AQI)')

plt.tight_layout()
plt.show()


## 15. Data-Driven AI Insights
All findings below are strictly calculated from the underlying dataset without external hallucination.

In [ ]:
# 1. Seasonal Increase Calculation
season_means = feat_df.groupby('Season')['AQI'].mean()
winter_val = season_means['Winter']
monsoon_val = season_means['Monsoon']
seasonal_surge_pct = ((winter_val - monsoon_val) / monsoon_val) * 100

# 2. City Hotspot Calculation
city_means = df_cleaned.groupby('City')['AQI'].mean().sort_values(ascending=False)
top_city = city_means.index[0]
top_city_val = city_means.iloc[0]
nat_median = df_cleaned['AQI'].median()
city_elevation_pct = ((top_city_val - nat_median) / nat_median) * 100

# 3. Leading Pollutant Correlation
p_corrs = df_cleaned[['AQI'] + pol_list].corr()['AQI'].drop('AQI').sort_values(ascending=False)
top_pol = p_corrs.index[0]
top_pol_r = p_corrs.iloc[0]

print("=== AUTOMATED DATA-DRIVEN INSIGHTS ===")
print(f"1. [Seasonal Inversion Surge]: Winter AQI averages {winter_val:.1f}, which is {seasonal_surge_pct:.1f}% higher than Monsoon ({monsoon_val:.1f}).")
print(f"2. [Spatial Pollution Hotspot]: {top_city} recorded the highest historical average AQI ({top_city_val:.1f}), exceeding the national median by {city_elevation_pct:.1f}%.")
print(f"3. [Primary Chemical Driver]: {top_pol} demonstrates the strongest linear correlation with composite AQI (Pearson r = {top_pol_r:.3f}).")


## 16. Conclusion
This project successfully designed and implemented an end-to-end Air Quality Analytics and 24-Hour Ahead Prediction System:
1. **Analytical Findings**: Verified that particulate matter ($PM_{2.5}, PM_{10}$) overwhelmingly dictates the National Air Quality Index in Indian urban centers. Seasonal atmospheric inversions during winter months create recurring, statistically significant surges in pollution concentration.
2. **Predictive Performance**: Evaluated multiple architectures under rigorous chronological partitioning. Ensemble tree architectures (Random Forest / XGBoost) outperform linear regression and standard persistence baselines by capturing non-linear meteorological dynamics.
3. **Zero-Leakage Integrity**: Established a strict temporal feature engineering protocol, guaranteeing that future observations never contaminate historical rolling aggregations.


## 17. Limitations
* **Regional vs. Microclimate Resolution**: Station data reflects ambient regional background air quality. It does not account for street-level microclimatic dispersion or hyperlocal traffic corridor peaks.
* **Absence of Real-Time Meteorology**: The dataset lacks high-resolution boundary-layer meteorology (wind vector fields, planetary boundary layer height, temperature inversion indices, and relative humidity), which are critical physical drivers of dispersion.
* **Exogenous Shock Unpredictability**: Unannounced episodic emissions (e.g. firecracker bursts or sudden agricultural residue fires) cannot be anticipated solely from autoregressive historical features.
* **Academic Disclaimer**: Predictions are produced by a student academic model for educational purposes and do not constitute official regulatory advisories from the CPCB.


## 18. Future Scope & References

### Future Scope:
1. **Multimodal Satellite Integration**: Incorporate Sentinel-5P TROPOMI and MODIS aerosol optical depth (AOD) satellite data.
2. **Deep Sequence Modeling**: Implement Temporal Fusion Transformers (TFT) or Long Short-Term Memory (LSTM) networks with attention mechanisms.
3. **Meteorological Fusion**: Integrate real-time ECMWF / ERA5 weather reanalysis data to improve dispersion modeling.

### References:
1. Central Pollution Control Board (CPCB), Ministry of Environment, Forest & Climate Change, Government of India. *National Air Quality Index Guidelines* (2014).
2. World Health Organization (WHO). *WHO Global Air Quality Guidelines: Particulate Matter, Ozone, Nitrogen Dioxide, Sulfur Dioxide and Carbon Monoxide* (2021).
3. Guttikunda, S. K., & Goel, R. (2013). Health impacts of particulate pollution in a megacity—Delhi, India. *Environmental Development*, 6, 8-20.
4. Rao, Rohan. *Air Quality Data in India (2015-2020)*, Kaggle Public Repository.
